# Vibrato benchmark

This notebook exercises the end-to-end benchmark described in `README.md`: known vibrato curves are injected into monophonic CocoChorales MIDI stems, rendered with the benchmark-only sfizz straight-tone policy, recovered through Attune's cached pYIN + smoothing stage, and compared with the source-faithful AVA ports and the other primary estimators. The scoring crop is never passed to an estimator.

Two suites keep the common and difficult conditions easy to run separately:

- **Constant/control:** clean, steady vibrato whose rate is seeded uniformly over 3--10 Hz and whose one-sided extent is seeded uniformly over 0.15--1.00 semitones, plus a straight-pitch control.
- **Changing:** seeded accelerating, decelerating, widening, and narrowing profiles under clean, 20 dB, and 10 dB conditions, plus the same straight-pitch control. Rate changes span 1--3 Hz; one-sided extent changes span 0.15--0.50 semitones. Every curve stays within the 3--10 Hz and 0.15--1.00-semitone product support.

The bounded Coco selection fixes ensemble and instrument quotas, then uses `CONFIG.seed` only to choose concrete tracks/stems inside each stratum. At the 20-stem preliminary size this means five brass, five random, five string, and five woodwind stems. Parameter draws are keyed by the same global seed plus stable track, note-index, and profile identity. Results and artifacts use synthesis-policy/sampling-version/seed-specific names, so reports rendered by another synthesizer cannot be reused accidentally. Extent and Rate precision/recall/F1 require the method's final positive detection and an estimate within the configured parameter tolerance. Wrong-valued positive predictions count as both FP and FN, while predictions on straight-pitch frames count as FP. Overall precision, recall, and F1 are the corresponding macro averages across extent and rate. Binary Detection Accuracy and Attune-only pitch-center accuracy remain separate diagnostics.


Run all reuses completed reports by default. Extent Accuracy (Yang) and Rate Accuracy (Yang) report mean matched-vibrato relative accuracy on a 0–1 scale; unmatched intervals are excluded. These columns use interval averages, unlike the frame-level tolerance-matched F1 columns. Set `force_preliminary_rerun=True` in `NotebookConfig` to rerun the benchmarks.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

REPO_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "app.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))

from benchmarks.modules.vibrato.VibratoNotebook import NotebookConfig, VibratoNotebook

In [3]:
CONFIG = NotebookConfig()
BENCH = VibratoNotebook(CONFIG, repo_root=REPO_ROOT)
BENCH.show_configuration()

,setting,value
0,runner,/Users/sarah/Desktop/shasha/Attune/benchmarks/...
1,sfizz renderer,/Users/sarah/dev/sfizz-1.2.3/build/library/bin...
2,soundfonts,/Users/sarah/Desktop/shasha/Attune/resources/s...
3,workers,9
4,preliminary stems,20
5,random seed,0
6,result version,primary_v7
7,Attune curve knot spacing,0.1 s
8,pYIN/HMM range,exact annotated F0 range
9,YIN window,"automatic: 4 guarded periods, power of two, mi..."


,setting,value
0,runner,/Users/sarah/Desktop/shasha/Attune/benchmarks/...
1,sfizz renderer,/Users/sarah/dev/sfizz-1.2.3/build/library/bin...
2,soundfonts,/Users/sarah/Desktop/shasha/Attune/resources/s...
3,workers,9
4,preliminary stems,20
5,random seed,0
6,result version,primary_v7
7,Attune curve knot spacing,0.1 s
8,pYIN/HMM range,exact annotated F0 range
9,YIN window,"automatic: 4 guarded periods, power of two, mi..."


## Straight-tone synthesis control

This renders one default violin A4 without injected pitch bend, runs both raw and production-smoothed Attune pYIN, and fails immediately if the production vibrato estimator finds any vibrato. The table also exposes voiced coverage and residual pitch spread so synthesis drift is visible before the larger benchmark runs.

In [4]:
sfizz_control_report = BENCH.run_sfizz_control()

Reusing completed reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/vibrato_runs/zero_vibrato_control_coco_sfizz_straight_tone_v1


,pitch_stage,detected_any_vibrato,detected_frames,scored_frames,voiced_frames,voiced_fraction,max_reported_rate_hz,max_reported_width_cents,pitch_p95_minus_p05_cents
0,raw,False,0,1018,1018,1.0000,0.0000,0.0000,8.8305
1,smoothed,False,0,1018,1018,1.0000,0.0000,0.0000,10.0000


,instrument,pitch_midi,audio,library,sfz,renderer_identity
0,violin,69,/Users/sarah/Desktop/shasha/Attune/benchmarks/...,SSO,sso-4.0/Sonatina Symphonic Orchestra/Strings -...,04c7564a24a2d5f2f0eb3f269fb328d99bda42464581d3...


## Sampling invariant and preliminary constant/control comparison

The first cell verifies stable ensemble/instrument quotas across seeds and reproducible parameter draws. The second runs the production Attune path on the balanced constant/control suite. Coco pitch analysis uses automatic power-of-two YIN windows sized for four periods at the score minimum after the production ±8-semitone guard.

In [5]:
selection_table, parameter_table = BENCH.show_sampling_invariants()

,seed,ensemble,instrument,stems
0,0,brass,horn,2
1,0,brass,trombone,1
2,0,brass,trumpet,1
3,0,brass,tuba,1
4,0,random,cello,1
5,0,random,clarinet,1
6,0,random,flute,1
7,0,random,oboe,1
8,0,random,violin,1
9,0,string,cello,1


,profile,rate_start_hz,rate_middle_hz,rate_end_hz,amplitude_start_semitones,amplitude_end_semitones
0,constant,8.9110,8.9110,8.9110,0.7943,0.7943
1,none,0.0000,0.0000,0.0000,0.0000,0.0000
2,accelerating,3.4996,4.0562,4.6127,0.9557,0.9557
3,decelerating,7.9166,7.0466,6.1767,0.6126,0.6126
4,widening,4.6523,4.6523,4.6523,0.4218,0.5950
5,narrowing,7.3603,7.3603,7.3603,0.9657,0.5568


In [6]:
preliminary_average = BENCH.run_preliminary_average()

Reusing completed reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/vibrato_runs/preliminary_average_coco_sfizz_straight_tone_v1__seeded_proportional_ensemble_instrument_v1__seeded_bounded_uniform_v1_primary_v7_seed0_auto_yin_4periods_pow2


,Method,Overall F1,Overall Precision,Overall Recall,Extent F1,Extent Precision,Extent Recall,Extent Accuracy (Yang),Rate F1,Rate Precision,...,Rate Accuracy (Yang),Center Accuracy (Attune),Detection F1,Detection Precision,Detection Recall,Detection Accuracy,False Alarms,Skipped,Errors,Audio(s)/Compute(s)
0,rossignol_extrema,0.4734,0.7854,0.3388,0.3491,0.5793,0.2499,0.9078,0.5977,0.9916,...,0.9587,NaN,0.5985,0.9930,0.4283,0.6912,0.0035,0,0,2042.2944
1,herrera_bonada_stft,0.2450,0.5357,0.1588,0.0732,0.1601,0.0475,0.8791,0.4167,0.9114,...,0.9554,NaN,0.4572,1.0000,0.2964,0.6219,0.0000,0,0,5474.4920
2,ventura_sousa_ferreira,0.4492,0.4463,0.4530,0.4146,0.4249,0.4047,0.6985,0.4838,0.4676,...,0.6805,NaN,0.6300,0.6089,0.6526,0.5881,0.4868,0,0,196.9093
3,driedger_template,0.5332,0.8921,0.3803,0.4857,0.8125,0.3463,0.9173,0.5808,0.9717,...,0.9160,NaN,0.5977,1.0000,0.4263,0.6917,0.0000,0,0,10.2488
4,driedger_template_benchmark_range,0.5974,0.4606,0.8498,0.5444,0.4197,0.7744,0.9781,0.6504,0.5014,...,0.9861,NaN,0.6917,0.5333,0.9840,0.5287,1.0000,0,0,3.3219
5,yang_ava_fdm,0.3093,0.3768,0.2623,0.2605,0.3174,0.2209,0.8045,0.3580,0.4362,...,0.8646,NaN,0.8208,1.0000,0.6960,0.8367,0.0000,0,0,22.9250
6,yang_ava_fdm_br,0.1537,0.3845,0.0960,0.1709,0.4276,0.1068,0.7578,0.1365,0.3414,...,0.8379,NaN,0.3859,0.9655,0.2412,0.5876,0.0100,0,0,22.2933
7,mcleod_tartini_prony,0.5545,0.5166,0.5985,0.4270,0.3978,0.4609,0.7238,0.6820,0.6353,...,0.8326,NaN,0.7955,0.7411,0.8586,0.7628,0.3484,0,0,342.7258
8,attune,0.6995,0.6733,0.7277,0.4993,0.4807,0.5195,0.8129,0.8996,0.8660,...,0.9544,0.8446,0.9302,0.8955,0.9678,0.9220,0.1311,0,0,62.3722


### Optional diagnostic: no-vibrato false positives

This audit treats every straight-tone case as a negative control. It reports frame-weighted and case-level false-positive rates, contiguous run duration, note location, raw/smoothed pitch error, octave errors, HMM-transition overlap, estimated rate/extent, and instrument/window breakdowns. Diagnostic patterns are descriptive evidence, not extra detector gates.

In [7]:
# Uncomment when investigating false positives.
# no_vibrato = BENCH.show_no_vibrato_analysis(preliminary_average)

### Optional diagnostic: pYIN range and automatic-window audit

This cell asserts, per selected stem, that both the autocorrelation search and HMM grid cover eight semitones below and above the annotated pitch range. It also recomputes every automatic window choice and requires a mixed distribution containing low-register windows of at least 4096 samples.

In [8]:
# Uncomment when auditing pitch-range/window behavior.
# pitch_geometry = BENCH.show_pitch_geometry(preliminary_average)

## Preliminary Yang real-audio parameter comparison

Runs one erhu and one violin recording from the released `Areas_and_parameters` subset. Only the Huangjiangqin and Yangjian recordings include the manual half-cycle marks required for rate and extent ground truth; the other performers support area detection only. Following Yang et al., consecutive manual extrema define half-cycle rate and peak-to-trough extent. Because the release omits pYIN tracks for these two performers, the adapter computes one common HMM-smoothed Attune pYIN contour and caches it.

This is a preliminary parameter check, not a fair false-positive or note-boundary comparison: the annotated vibrato spans are supplied as pseudo-note boundaries to note-aware methods, and the subset does not provide the explicit negative controls present in the Coco suites. It uses the same tolerance-matched extent/rate precision, recall, and F1 implementation, but precision here should not be read as full false-alarm coverage. The AVA Bayes model also has training provenance in this corpus, so treat its row as exploratory rather than held-out evidence.

In [9]:
preliminary_yang = BENCH.run_yang_preliminary()

Reusing completed reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/vibrato_runs/preliminary_yang_parameters_primary_v7


,Method,Overall F1,Overall Precision,Overall Recall,Extent F1,Extent Precision,Extent Recall,Extent Accuracy (Yang),Rate F1,Rate Precision,...,Rate Accuracy (Yang),Center Accuracy (Attune),Detection F1,Detection Precision,Detection Recall,Detection Accuracy,False Alarms,Skipped,Errors,Audio(s)/Compute(s)
0,rossignol,0.1712,0.4490,0.1058,0.0361,0.0946,0.0223,0.5767,0.3064,0.8034,...,0.9618,NaN,0.3814,1.0000,0.2356,0.2356,NaN,0,0,2662.0221
1,herrera_bonada,0.0443,0.3946,0.0235,0.0399,0.3554,0.0211,0.7486,0.0487,0.4338,...,0.8604,NaN,0.1122,1.0000,0.0595,0.0595,NaN,0,0,6105.1460
2,ventura_sousa_ferreira,0.4010,0.4337,0.3730,0.3455,0.3737,0.3213,0.2429,0.4566,0.4937,...,0.8164,NaN,0.9247,1.0000,0.8599,0.8599,NaN,0,0,191.6041
3,driedger,0.1861,0.4976,0.1144,0.0000,0.0000,0.0000,0.0626,0.3722,0.9953,...,0.9734,NaN,0.3739,1.0000,0.2300,0.2300,NaN,0,0,10.9638
4,driedger_benchmark_range,0.3882,0.4269,0.3559,0.2923,0.3214,0.2680,0.1022,0.4841,0.5323,...,0.8634,NaN,0.9094,1.0000,0.8338,0.8338,NaN,0,0,3.1864
5,yang_dt,0.1823,0.3183,0.1277,0.2154,0.3760,0.1509,0.7110,0.1492,0.2606,...,0.9328,NaN,0.5741,1.0000,0.4026,0.4026,NaN,0,0,15.5492
6,yang_br,0.2830,0.5014,0.1971,0.4045,0.7167,0.2817,0.8678,0.1614,0.2860,...,0.9321,NaN,0.5668,1.0000,0.3955,0.3955,NaN,0,0,15.3549
7,mcleod,0.4478,0.4897,0.4126,0.4877,0.5333,0.4493,0.4311,0.4080,0.4461,...,0.6970,NaN,0.9145,1.0000,0.8425,0.8425,NaN,0,0,193.8350
8,attune,0.6262,0.6262,0.6262,0.5810,0.5810,0.5810,0.7794,0.6715,0.6715,...,0.8984,0.8914,1.0000,1.0000,1.0000,1.0000,NaN,0,0,100.8294


## Preliminary changing-profile comparison

Runs the same 20-stem balanced selection with accelerating, decelerating, widening, narrowing, and straight-pitch profiles. Fixed quantities are seeded across the full 3--10 Hz and 0.15--1.00-semitone support. Within-note rate spans are limited to 1--3 Hz and one-sided extent spans to 0.15--0.50 semitones. Clean, 20 dB, and 10 dB versions exercise the complete audio-to-pitch-to-vibrato path. The run uses production Attune settings; previously rejected ablation settings remain disabled.

In [10]:
preliminary_changing = BENCH.run_preliminary_changing()

Reusing completed reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/vibrato_runs/preliminary_changing_coco_sfizz_straight_tone_v1__seeded_proportional_ensemble_instrument_v1__seeded_bounded_uniform_v1_primary_v7_seed0_auto_yin_4periods_pow2


,Method,Overall F1,Overall Precision,Overall Recall,Extent F1,Extent Precision,Extent Recall,Extent Accuracy (Yang),Rate F1,Rate Precision,...,Rate Accuracy (Yang),Center Accuracy (Attune),Detection F1,Detection Precision,Detection Recall,Detection Accuracy,False Alarms,Skipped,Errors,Audio(s)/Compute(s)
0,rossignol_extrema,0.5812,0.7866,0.4608,0.4299,0.5819,0.3409,0.8878,0.7324,0.9913,...,0.9676,NaN,0.7381,0.9991,0.5853,0.6648,0.0023,0,0,2015.2190
1,herrera_bonada_stft,0.4075,0.6215,0.3031,0.2202,0.3358,0.1638,0.8409,0.5948,0.9072,...,0.9030,NaN,0.6557,1.0000,0.4877,0.5865,0.0000,0,0,5167.7105
2,ventura_sousa_ferreira,0.5182,0.5459,0.4932,0.4178,0.4432,0.3951,0.8144,0.6187,0.6487,...,0.8267,NaN,0.7981,0.8368,0.7628,0.6885,0.6228,0,0,191.1674
3,driedger_template,0.4954,0.7141,0.3792,0.4171,0.6014,0.3193,0.9189,0.5736,0.8269,...,0.9517,NaN,0.6934,0.9997,0.5308,0.6211,0.0007,0,0,12.4282
4,driedger_template_benchmark_range,0.6980,0.6321,0.7791,0.5727,0.5187,0.6393,0.6553,0.8232,0.7456,...,0.8815,NaN,0.8902,0.8062,0.9938,0.8022,1.0000,0,0,3.3231
5,yang_ava_fdm,0.4016,0.4223,0.3828,0.2379,0.2502,0.2268,0.7809,0.5652,0.5944,...,0.8922,NaN,0.9518,0.9993,0.9086,0.9257,0.0026,0,0,21.1541
6,yang_ava_fdm_br,0.1716,0.4863,0.1042,0.1352,0.3832,0.0821,0.6530,0.2079,0.5893,...,0.8737,NaN,0.3490,0.9889,0.2119,0.3619,0.0099,0,0,19.6758
7,mcleod_tartini_prony,0.6262,0.6200,0.6326,0.4789,0.4742,0.4838,0.7612,0.7735,0.7659,...,0.8680,NaN,0.9033,0.8943,0.9124,0.8423,0.4514,0,0,295.3985
8,attune,0.7036,0.6978,0.7096,0.5014,0.4973,0.5057,0.8094,0.9058,0.8983,...,0.9480,0.9234,0.9862,0.9780,0.9945,0.9775,0.0937,0,0,51.3722


## Optional diagnostic: step-3 knot-spacing ablation

`vib2_curve_sec` sets the interior knot spacing of the cubic B-spline bases carrying Attune's center and width curves. An offline replay sweep identified it, not `vib2_width_smoothness`, as the control holding back changing-profile extent: the width penalty is already flat between `0.1` and `1.0` and falls off on both sides, so it remains at its `1.0` default, while spacings below the former `0.4` default improved extent F1 on the changing, constant/control, and Yang tiers at once. The winning `0.1 s` spacing is now the shipped default. A second-difference penalty on B-spline coefficients does not penalize a straight ramp, which is why it could never be what flattens a widening or narrowing profile; the gain comes from a center curve fine enough to follow intonation drift inside a note, leaving a cleaner residual for the sinusoid. Method and evidence are in `curve_penalty_sweep.md`.

Each value re-runs Attune only — no competitor reads `vib2_curve_sec` — against the changing suite above, reusing that suite's rendered audio and pitch cache, so the sweep costs detector time alone. Results land in their own `_curve_sec_*` directories and cannot overwrite or be mistaken for the shipped-default run. The shipped `0.1` row reproduces `preliminary_changing`'s `attune` row; `0.4` is retained as the historical baseline, and the competitor rows in `preliminary_changing` remain the comparison. Pass `suite="average"` to repeat the sweep on the constant/control safeguard, or `values=(...)` to change the grid.

`CONFIG.workers` sizes both pitch preparation and estimator-analysis process pools. Because each ablation runner therefore already uses parallel workers, the sweep runs its values sequentially by default to avoid nested oversubscription and to keep `Audio(s)/Compute(s)` interpretable. The first invocation also primes the shared render/pitch cache before any other value reads it. Pass `parallel=True` only for a deliberately contended stress run; remaining values will then execute concurrently after cache priming.

In [11]:
# The adopted spacing has already been validated; uncomment to repeat.
# curve_sec_ablation = BENCH.run_curve_sec_ablation()

## Full comparisons

The final two code cells remove the preliminary stem limit and run every stem in the configured CocoChorales test split. These are long, cache-backed runs. Jobs use `CONFIG.workers` spawned workers with one numerical-library thread per worker; sfizz subprocesses are deliberately serialized to avoid truncated renders, while pYIN work remains parallel. Vibrato estimators still receive each complete detected stem and shared MIDI note-boundary set.

In [12]:
# all_average = BENCH.run_all_average()

In [13]:
# all_changing = BENCH.run_all_changing()